In [8]:
import numpy as np


def clip_segment_to_box_4d(p0, p1, edges, eps=1e-12):
    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    box_min = np.array([e[0] for e in edges])
    box_max = np.array([e[-1] for e in edges])

    t0, t1 = 0.0, 1.0

    for i in range(4):
        if abs(d[i]) < eps:
            if p0[i] < box_min[i] or p0[i] > box_max[i]:
                return None
            continue

        inv = 1.0 / d[i]
        ta = (box_min[i] - p0[i]) * inv
        tb = (box_max[i] - p0[i]) * inv

        if ta > tb:
            ta, tb = tb, ta

        t0 = max(t0, ta)
        t1 = min(t1, tb)

        if t0 > t1:
            return None

    return t0, t1


def voxel_traversal_4d(p0, p1, edges, eps=1e-12):
    """
    Yields for each crossed voxel:
    (index_tuple, entry_point, exit_point)
    """

    p0 = np.asarray(p0, float)
    p1 = np.asarray(p1, float)
    d = p1 - p0

    edges = [np.asarray(e, float) for e in edges]

    clipped = clip_segment_to_box_4d(p0, p1, edges, eps)
    if clipped is None:
        return

    t_enter, t_exit = clipped

    # ---- Ensure starting voxel is included ----
    p_start = p0 + (t_enter + eps) * d

    # Initial voxel index
    idx = np.array(
        [np.searchsorted(edges[i], p_start[i], side="right") - 1 for i in range(4)],
        dtype=int,
    )

    step = np.sign(d).astype(int)

    # Compute next boundary crossings
    tNext = np.zeros(4)

    for i in range(4):
        if abs(d[i]) < eps:
            tNext[i] = np.inf
            continue

        if step[i] > 0:
            boundary = edges[i][idx[i] + 1]
        else:
            boundary = edges[i][idx[i]]

        tNext[i] = (boundary - p0[i]) / d[i]

    t = t_enter

    while (
        np.all(idx >= 0)
        and all(idx[i] < len(edges[i]) - 1 for i in range(4))
        and t <= t_exit + eps
    ):
        axis = np.argmin(tNext)
        t_voxel_exit = min(tNext[axis], t_exit)

        entry = p0 + t * d
        exit_ = p0 + t_voxel_exit * d

        yield tuple(idx), entry, exit_

        if tNext[axis] > t_exit:
            break

        # Step to next voxel
        idx[axis] += step[axis]
        t = tNext[axis]

        if idx[axis] < 0 or idx[axis] >= len(edges[axis]) - 1:
            break

        # Update next boundary crossing for this axis
        if step[axis] > 0:
            boundary = edges[axis][idx[axis] + 1]
        else:
            boundary = edges[axis][idx[axis]]

        tNext[axis] = (boundary - p0[axis]) / d[axis]

In [10]:
edges = [
    np.array([0.0, 0.5, 2.0, 3.0]),
    np.array([0.0, 1.0, 1.5, 4.0]),
    np.array([0.0, 0.2, 0.8, 1.0]),
    np.array([0.0, 2.0, 5.0]),
]

p0 = [-0.2, 0.3, 0.1, 0.5]
p1 = [2.8, 3.2, 0.9, 4.8]

for idx, p_in, p_out in voxel_traversal_4d(p0, p1, edges):
    print("voxel: ({},{},{},{})".format(*idx))
    print("  entry:", p_in)
    print("  exit :", p_out)

voxel: (0,0,0,0)
  entry: [0.         0.49333333 0.15333333 0.78666667]
  exit : [0.175  0.6625 0.2    1.0375]
voxel: (0,0,1,0)
  entry: [0.175  0.6625 0.2    1.0375]
  exit : [0.5        0.97666667 0.28666667 1.50333333]
voxel: (1,0,1,0)
  entry: [0.5        0.97666667 0.28666667 1.50333333]
  exit : [0.52413793 1.         0.29310345 1.53793103]
voxel: (1,1,1,0)
  entry: [0.52413793 1.         0.29310345 1.53793103]
  exit : [0.84651163 1.31162791 0.37906977 2.        ]
voxel: (1,1,1,1)
  entry: [0.84651163 1.31162791 0.37906977 2.        ]
  exit : [1.04137931 1.5        0.43103448 2.27931034]
voxel: (1,2,1,1)
  entry: [1.04137931 1.5        0.43103448 2.27931034]
  exit : [2.         2.42666667 0.68666667 3.65333333]
voxel: (2,2,1,1)
  entry: [2.         2.42666667 0.68666667 3.65333333]
  exit : [2.425  2.8375 0.8    4.2625]
voxel: (2,2,2,1)
  entry: [2.425  2.8375 0.8    4.2625]
  exit : [2.8 3.2 0.9 4.8]


In [11]:
p0 = [0.1, 0.3, 0.1, 0.5]
p1 = [0.2, 0.3, 0.1, 0.5]

for idx, p_in, p_out in voxel_traversal_4d(p0, p1, edges):
    print("voxel: ({},{},{},{})".format(*idx))
    print("  entry:", p_in)
    print("  exit :", p_out)

voxel: (0,0,0,0)
  entry: [0.1 0.3 0.1 0.5]
  exit : [0.2 0.3 0.1 0.5]
